# 04 Fair Comparison of an Ordinary LLM and the EC-LLM

This notebook uses the same condition-specific evidence packages as notebook `03`. API calls are disabled by default; see `Code/README.md` for path, offline-evaluation, and API configuration.

In [ ]:
# If dependencies are missing, uncomment and run the following line.
# %pip install openai pandas numpy openpyxl matplotlib

from pathlib import Path
from datetime import datetime
import getpass
import hashlib
import json
import math
import os
import random
import re
import time
import uuid

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT_OVERRIDE = os.getenv("ROBOT_ACTUATOR_REPO_ROOT") or None
EVIDENCE_DIR_OVERRIDE = os.getenv("ROBOT_ACTUATOR_EVIDENCE_DIR") or None
OUTPUT_DIR_OVERRIDE = os.getenv("ROBOT_ACTUATOR_LLM_OUTPUT_DIR") or None

def resolve_repo_root(override=None):
    if override:
        root = Path(override).expanduser().resolve()
        if not (root / "Code").is_dir() or not (root / "Data").is_dir():
            raise FileNotFoundError(f"Invalid repository root: {root}")
        return root
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Code").is_dir() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError(
        "Unable to locate the repository root automatically. Start Jupyter from the repository root or Code/, "
        "or set ROBOT_ACTUATOR_REPO_ROOT."
    )

REPO_ROOT = resolve_repo_root(REPO_ROOT_OVERRIDE)
EVIDENCE_DIR = (
    Path(EVIDENCE_DIR_OVERRIDE).expanduser().resolve()
    if EVIDENCE_DIR_OVERRIDE
    else REPO_ROOT / "Data" / "Evidence_Packages"
)
OUTPUT_DIR = (
    Path(OUTPUT_DIR_OVERRIDE).expanduser().resolve()
    if OUTPUT_DIR_OVERRIDE
    else REPO_ROOT / "Reproduced_Outputs" / "04_fair_llm_comparison"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Safe default: set to True only after verifying the configuration, API access, and expected cost.
RUN_LLM_CALLS = False
DOGAPI_BASE_URL = "https://www.dogapi.cc/v1"
DOGAPI_MODEL = "gpt-5.6-luna"
TEMPERATURE = 0.2
TOP_P = 1.0
MAX_TOKENS = 1600
REQUEST_TIMEOUT = 120
N_REPEATS = 5
NETWORK_RETRIES = 2
NETWORK_BACKOFF_SECONDS = 8
GROUPS = ["ordinary_llm", "ec_llm"]
RANDOM_STATE = 42
HUMAN_SAMPLES_PER_CONDITION_GROUP = 2
RATER_IDS = ["R1", "R2", "R3"]

package_paths = sorted(EVIDENCE_DIR.glob("03_xgboost_shap_physics_score_*.json"))
if not package_paths:
    raise FileNotFoundError(
        f"No files matching 03_xgboost_shap_physics_score_*.json were found in {EVIDENCE_DIR}."
        "Place the complete evidence-package set in that directory or set ROBOT_ACTUATOR_EVIDENCE_DIR."
    )
packages = [json.loads(path.read_text(encoding="utf-8")) for path in package_paths]
if len(packages) != 8:
    print(f"Warning: {len(packages)} task conditions were detected; the manuscript experiment specifies eight. Verify the evidence-package directory.")

expected_total = len(packages) * len(GROUPS) * N_REPEATS
print("Evidence-package directory:", EVIDENCE_DIR)
print("Output directory:", OUTPUT_DIR)
print(f"Planned generation: {len(packages)} conditions x {len(GROUPS)} groups x {N_REPEATS} repeats = {expected_total} single-pass generations")
print("Model and sampling parameters:", DOGAPI_MODEL, TEMPERATURE, TOP_P, MAX_TOKENS)
print("RUN_LLM_CALLS =", RUN_LLM_CALLS)

In [ ]:
EVIDENCE_PATTERN = re.compile(r"\b(?:REQ|ML|SHAP|PHY|SCORE|RULE|EXP)-[A-Za-z0-9_.\-/]+")
RISK_PATTERN = re.compile(r"\bRISK-[A-Za-z0-9_.\-/]+")
LOW_MARGIN_RISK_THRESHOLD = 1.30
MARGIN_NEAR_REQUIRED_RATIO = 1.10

def walk(obj):
    if isinstance(obj, dict):
        for key, value in obj.items():
            yield key, value
            yield from walk(value)
    elif isinstance(obj, list):
        for item in obj:
            yield from walk(item)

def collect_evidence_ids(obj):
    ids = set()
    for key, value in walk(obj):
        if key in {"evidence_id", "evidence_ids", "evidence_refs"}:
            if isinstance(value, str):
                ids.add(value)
            elif isinstance(value, list):
                ids.update(str(x) for x in value)
        if isinstance(value, str):
            ids.update(EVIDENCE_PATTERN.findall(value))
    return {x for x in ids if x and x.lower() != "none"}

def find_candidate(package, candidate_id):
    for key in ("candidate_records", "ranked_candidates"):
        for rec in package.get(key, []) or []:
            if rec.get("candidate_id") == candidate_id:
                return rec
    return None

def finite_float(value):
    try:
        value = float(value)
        return value if math.isfinite(value) else None
    except Exception:
        return None

def compact_candidate(rec):
    if not isinstance(rec, dict):
        return None
    physics = rec.get("physics_check", {}) or {}
    score = rec.get("multi_objective_score", {}) or {}
    specs = rec.get("specs", {}) or {}
    return {
        "candidate_id": rec.get("candidate_id"),
        "drive_type": rec.get("drive_type"),
        "manufacturer": rec.get("manufacturer"),
        "model": rec.get("model"),
        "specs": {
            "output_angle_deg": specs.get("output_angle_deg", specs.get("\u8f93\u51fa\u89d2\u5ea6\uff08deg\uff09")),
            "rated_speed_rpm": specs.get("rated_speed_rpm", specs.get("\u989d\u5b9a\u8f6c\u901f(rpm)")),
            "peak_speed_rpm": specs.get("peak_speed_rpm", specs.get("\u5cf0\u503c\u8f6c\u901f(rpm)")),
            "rated_torque_Nm": specs.get("rated_torque_Nm", specs.get("\u989d\u5b9a\u529b\u77e9(N\u00b7m)")),
            "peak_torque_Nm": specs.get("peak_torque_Nm", specs.get("\u5cf0\u503c\u529b\u77e9(N\u00b7m)")),
            "rated_power_W": specs.get("rated_power_W", specs.get("\u989d\u5b9a\u529f\u7387\uff08W\uff09")),
            "peak_power_W": specs.get("peak_power_W", specs.get("\u5cf0\u503c\u529f\u7387\uff08W\uff09")),
            "mass_kg": specs.get("mass_kg", specs.get("\u8d28\u91cf\uff08kg\uff09")),
            "volume_ml": specs.get("volume_ml", specs.get("\u4f53\u79ef\uff08ml\uff09")),
            "efficiency": specs.get("efficiency_fraction", specs.get("efficiency", specs.get("\u6548\u7387\uff08%\uff09"))),
        },
        "physics_check": {
            "evidence_id": physics.get("evidence_id"),
            "result": physics.get("result") or physics.get("status"),
            "overall_safety_margin": physics.get("overall_safety_margin"),
            "limiting_constraint": physics.get("limiting_constraint"),
            "limiting_constraint_label": physics.get("limiting_constraint_label"),
            "conditional_constraints": physics.get("conditional_constraints") or [],
            "fail_constraints": physics.get("fail_constraints") or [],
            "comparisons": physics.get("comparisons") or {},
        },
        "multi_objective_score": {
            "evidence_id": score.get("evidence_id"),
            "rank": score.get("score_rank", score.get("rank")),
            "score": score.get("score"),
            "normalized_terms": score.get("normalized_terms", score.get("components")),
        },
    }

def safe_risk_suffix(value):
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(value or "unknown")).strip("_") or "unknown"

def risk_item(item_id, label, description, evidence_ids, severity="medium"):
    return {
        "risk_item_id": item_id,
        "label": label,
        "description": description,
        "severity": severity,
        "evidence_ids": sorted(set(str(x) for x in evidence_ids if x)),
    }

def build_required_risk_items(package):
    seed = package.get("recommendation_seed", {}) or {}
    selected = find_candidate(package, seed.get("candidate_id"))
    if not selected:
        return []
    physics = selected.get("physics_check", {}) or {}
    comparisons = physics.get("comparisons", {}) or {}
    evidence_ids = sorted(collect_evidence_ids(seed) | collect_evidence_ids(physics) | collect_evidence_ids(selected.get("multi_objective_score", {})))
    items = []

    limiting = physics.get("limiting_constraint")
    limiting_label = physics.get("limiting_constraint_label") or limiting
    if limiting:
        items.append(risk_item(
            f"RISK-LIMIT-{safe_risk_suffix(limiting)}",
            f"Limiting physical constraint: {limiting_label}",
            "This constraint determines the recommended candidate's minimum physical safety margin and should receive priority in follow-up verification.",
            evidence_ids, "high",
        ))

    for name in physics.get("conditional_constraints", []) or []:
        label = (comparisons.get(name, {}) or {}).get("label") or str(name)
        items.append(risk_item(
            f"RISK-CONDITIONAL-{safe_risk_suffix(name)}",
            f"Conditionally satisfied constraint: {label}",
            "This constraint does not meet the recommended safety factor and should be reassessed with the applicable duty duration, control strategy, and thermal boundary conditions.",
            evidence_ids, "high",
        ))

    for name in physics.get("fail_constraints", []) or []:
        label = (comparisons.get(name, {}) or {}).get("label") or str(name)
        items.append(risk_item(
            f"RISK-FAIL-{safe_risk_suffix(name)}",
            f"Failed constraint: {label}",
            "This constraint fails the physical verification and requires either rejection of the candidate or a revised actuator selection.",
            evidence_ids, "critical",
        ))

    overall = finite_float(physics.get("overall_safety_margin"))
    if overall is not None and overall <= LOW_MARGIN_RISK_THRESHOLD:
        items.append(risk_item(
            "RISK-LOW-OVERALL-SAFETY-MARGIN",
            f"Low overall safety margin: {overall:.3g}",
            "The overall safety margin is close to the acceptance threshold; additional thermal, fatigue, or impact verification is required.",
            evidence_ids, "high",
        ))

    for name, comp in comparisons.items():
        comp = comp or {}
        actual = finite_float(comp.get("actual_margin"))
        required = finite_float(comp.get("required_margin"))
        if actual is not None and required is not None and actual <= required * MARGIN_NEAR_REQUIRED_RATIO:
            label = comp.get("label") or str(name)
            items.append(risk_item(
                f"RISK-NEAR-THRESHOLD-{safe_risk_suffix(name)}",
                f"Near-threshold constraint: {label}",
                f"The actual margin ({actual:.3g}) is close to the required margin ({required:.3g}); follow-up simulation or prototype testing is warranted.",
                evidence_ids, "medium",
            ))

    top_types = package.get("xgboost_prediction", {}).get("top_k_drive_types", []) or []
    top1 = top_types[0] if top_types else {}
    if seed.get("drive_type") and top1.get("drive_type") and seed.get("drive_type") != top1.get("drive_type"):
        mismatch_ids = sorted(set(evidence_ids) | collect_evidence_ids(top1))
        items.append(risk_item(
            "RISK-ML-TOP1-FINAL-MISMATCH",
            f"The XGBoost top-1 class is {top1.get('drive_type')}, whereas the final recommendation is {seed.get('drive_type')}",
            "The highest-similarity machine-learning class differs from the final recommendation; the explanation should identify the roles of physical verification and multi-objective scoring.",
            mismatch_ids, "medium",
        ))

    return list({item["risk_item_id"]: item for item in items}.values())

def build_evidence_object(package, top_n_ranked=5):
    seed = package.get("recommendation_seed", {}) or {}
    ranked = package.get("ranked_candidates", []) or []
    risks = build_required_risk_items(package)
    obj = {
        "evidence_object_version": "EC-LLM-fair-comparison-v1",
        "condition": package.get("joint_requirements", {}).get("condition"),
        "E_req": package.get("joint_requirements"),
        "E_ml": package.get("xgboost_prediction"),
        "E_phy_policy": package.get("physics_check_policy"),
        "E_score_policy": package.get("multi_objective_score_policy"),
        "E_recommendation_seed": seed,
        "E_selected_candidate": compact_candidate(find_candidate(package, seed.get("candidate_id"))),
        "E_ranked_candidates_top": [compact_candidate(rec) for rec in ranked[:top_n_ranked]],
        "E_required_risk_items": risks,
        "allowed_risk_item_ids": [item["risk_item_id"] for item in risks],
    }
    obj["allowed_evidence_ids"] = sorted(collect_evidence_ids(obj))
    return obj

evidence_objects = [build_evidence_object(package) for package in packages]
display(pd.DataFrame({
    "condition": [obj["condition"] for obj in evidence_objects],
    "allowed_evidence_ids": [len(obj["allowed_evidence_ids"]) for obj in evidence_objects],
    "required_risk_items": [len(obj["E_required_risk_items"]) for obj in evidence_objects],
}))


In [ ]:
REPORT_SCHEMA = {
    "recommended_candidate": {
        "candidate_id": "string", "drive_type": "string", "manufacturer": "string", "model": "string",
        "recommendation_level": "primary | backup | not_recommended | insufficient_evidence",
        "evidence_ids": ["evidence_id"],
    },
    "decision_reasons": [{"claim": "string", "evidence_ids": ["evidence_id"]}],
    "tradeoffs": [{"tradeoff": "string", "evidence_ids": ["evidence_id"]}],
    "risks": [{"risk": "string", "risk_item_ids": ["risk_item_id"], "evidence_ids": ["evidence_id"]}],
    "next_checks": [{"check": "string", "risk_item_ids": ["risk_item_id"], "evidence_ids": ["evidence_id"]}],
    "concise_report_cn": "string",
}

COMMON_SYSTEM = "You are an explanation system for robot-joint actuator selection. Organize the supplied computational results without rerunning classification, physical calculations, or multi-objective scoring."
COMMON_TASK = """
Using only the Evidence Object below, explain the recommended candidate, primary rationale, engineering trade-offs, risks, and follow-up verification.

Requirements shared by both groups:
1. Use English. Do not invent numerical values that are absent from the Evidence Object;
2. Do not alter the recommendation seed, candidate IDs, drive architectures, physical-verification status, or score ranking;
3. Return exactly one JSON object and no Markdown;
4. The top-level and nested fields must conform to the supplied schema, and each list-valued field must contain at least one item;
5. Use `primary` as the recommendation_level for the recommendation seed. When evidence is insufficient, state `insufficient evidence` explicitly. The legacy `concise_report_cn` field name is retained for schema compatibility but must contain English text.
""".strip()

GROUP_RULES = {
    "ordinary_llm": """
Ordinary-explanation condition: evidence_id and risk_item_id citations are optional, and the corresponding schema arrays may be empty.
Select risks according to conventional engineering-explanation practice; item-by-item coverage of E_required_risk_items is not required.
""".strip(),
    "ec_llm": """
Evidence-constrained condition: make factual statements only from the Evidence Object. The recommendation and every rationale, trade-off, risk, and follow-up check must cite valid evidence_id values.
Every item in E_required_risk_items must be covered in risks or next_checks and must include the corresponding risk_item_id and evidence_ids.
Do not cite identifiers outside allowed_evidence_ids or allowed_risk_item_ids.
""".strip(),
}

def canonical_json(obj):
    return json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":"))

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def build_prompts(evidence_object, group):
    # The Evidence Object, task text, and schema are byte-identical across groups; only GROUP_RULES differs.
    evidence_json = canonical_json(evidence_object)
    schema_json = canonical_json(REPORT_SCHEMA)
    user_prompt = (
        COMMON_TASK
        + "\n\nExperimental-group rule:\n" + GROUP_RULES[group]
        + "\n\nOutput schema:\n" + schema_json
        + "\n\nEvidence Object:\n" + evidence_json
    )
    return COMMON_SYSTEM, user_prompt, evidence_json, schema_json

# Fairness self-check: both groups must share the same Evidence Object, task, schema, and model parameters.
for obj in evidence_objects:
    p0 = build_prompts(obj, "ordinary_llm")
    p1 = build_prompts(obj, "ec_llm")
    assert p0[0] == p1[0]
    assert p0[2] == p1[2]
    assert p0[3] == p1[3]

print("Fairness self-check passed: both groups share the system message, Evidence Object, and schema; only the experimental-group rule differs.")


In [ ]:
TOP_LEVEL_FIELDS = ["recommended_candidate", "decision_reasons", "tradeoffs", "risks", "next_checks", "concise_report_cn"]

def strip_code_fence(text):
    text = str(text or "").strip()
    if text.startswith("```json"):
        text = text[len("```json"):]
    elif text.startswith("```"):
        text = text[3:]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()

def parse_report(text):
    cleaned = strip_code_fence(text)
    try:
        return json.loads(cleaned), "parsed"
    except Exception:
        first, last = cleaned.find("{"), cleaned.rfind("}")
        if first >= 0 and last > first:
            try:
                return json.loads(cleaned[first:last + 1]), "parsed_from_braces"
            except Exception as exc:
                return None, f"parse_error:{type(exc).__name__}"
        return None, "parse_error:no_json_object"

def id_list(value):
    if isinstance(value, str):
        return [value]
    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]
    return []

def schema_errors(report):
    if not isinstance(report, dict):
        return ["report_not_object"]
    errors = [f"missing:{name}" for name in TOP_LEVEL_FIELDS if name not in report]
    if not isinstance(report.get("recommended_candidate"), dict):
        errors.append("recommended_candidate_not_object")
    for name in ["decision_reasons", "tradeoffs", "risks", "next_checks"]:
        value = report.get(name)
        if not isinstance(value, list):
            errors.append(f"{name}_not_list")
        elif not value:
            errors.append(f"{name}_empty")
        elif not all(isinstance(item, dict) for item in value):
            errors.append(f"{name}_contains_non_object")
    if not isinstance(report.get("concise_report_cn"), str):
        errors.append("concise_report_cn_not_string")
    return errors

def claim_units(report):
    if not isinstance(report, dict):
        return []
    units = []
    rec = report.get("recommended_candidate")
    if isinstance(rec, dict):
        units.append(rec)
    for name in ["decision_reasons", "tradeoffs", "risks", "next_checks"]:
        units.extend(item for item in report.get(name, []) if isinstance(item, dict))
    return units

def report_used_evidence_ids(report, raw_text):
    return set(EVIDENCE_PATTERN.findall(json.dumps(report, ensure_ascii=False) if isinstance(report, dict) else str(raw_text or "")))

def report_used_risk_ids(report, raw_text):
    return set(RISK_PATTERN.findall(json.dumps(report, ensure_ascii=False) if isinstance(report, dict) else str(raw_text or "")))

def numerical_claim_values(report, evidence_object):
    skip_keys = {"candidate_id", "model", "manufacturer", "drive_type", "evidence_ids", "evidence_id", "risk_item_ids", "risk_item_id"}
    identifier_strings = set()
    for key, value in walk(evidence_object):
        if key in skip_keys and isinstance(value, str):
            identifier_strings.add(value)
    values = []
    def visit(obj, key=None):
        if key in skip_keys:
            return
        if isinstance(obj, dict):
            for k, value in obj.items():
                visit(value, k)
        elif isinstance(obj, list):
            for value in obj:
                visit(value, key)
        elif isinstance(obj, str):
            cleaned = EVIDENCE_PATTERN.sub(" ", obj)
            cleaned = RISK_PATTERN.sub(" ", cleaned)
            for identifier in sorted(identifier_strings, key=len, reverse=True):
                cleaned = cleaned.replace(identifier, " ")
            # Remove model names, candidate IDs, version strings, and similar alphanumeric identifiers so their digits are not treated as engineering values.
            cleaned = re.sub(r"(?i)\b(?=[A-Za-z0-9_./-]*[A-Za-z])(?=[A-Za-z0-9_./-]*\d)[A-Za-z0-9_./-]+\b", " ", cleaned)
            values.extend(float(x) for x in re.findall(r"(?<![A-Za-z])[-+]?\d+(?:\.\d+)?", cleaned))
        elif isinstance(obj, (int, float)) and not isinstance(obj, bool) and math.isfinite(float(obj)):
            values.append(float(obj))
    visit(report)
    return values

def all_numeric_evidence_values(obj):
    values = []
    def visit(value):
        if isinstance(value, dict):
            for item in value.values(): visit(item)
        elif isinstance(value, list):
            for item in value: visit(item)
        elif isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(float(value)):
            values.append(float(value))
    visit(obj)
    return values

def number_supported(value, allowed):
    # Treat 0.92 and 92% as equivalent representations and ignore standalone ordinal values of 0 or 1.
    if value in (0.0, 1.0):
        return True
    candidates = [value, value / 100.0, value * 100.0]
    for candidate in candidates:
        for reference in allowed:
            if math.isclose(candidate, reference, rel_tol=5e-3, abs_tol=1e-6):
                return True
    return False

def verify_report(report, raw_text, evidence_object, parse_status):
    allowed_evidence = set(evidence_object.get("allowed_evidence_ids", []))
    allowed_risks = set(evidence_object.get("allowed_risk_item_ids", []))
    used_evidence = report_used_evidence_ids(report, raw_text)
    used_risks = report_used_risk_ids(report, raw_text)
    invalid_evidence = sorted(used_evidence - allowed_evidence)
    invalid_risks = sorted(used_risks - allowed_risks)

    units = claim_units(report)
    supported_units = 0
    for unit in units:
        ids = set(id_list(unit.get("evidence_ids")) + id_list(unit.get("evidence_id")))
        if ids & allowed_evidence:
            supported_units += 1
    evidence_coverage = supported_units / len(units) if units else 0.0

    schema_errs = schema_errors(report)
    schema_completeness = sum(field in report for field in TOP_LEVEL_FIELDS) / len(TOP_LEVEL_FIELDS) if isinstance(report, dict) else 0.0
    seed = evidence_object.get("E_recommendation_seed", {}) or {}
    rec = report.get("recommended_candidate", {}) if isinstance(report, dict) else {}
    if not isinstance(rec, dict):
        rec = {}
    recommendation_consistency = float(bool(seed.get("candidate_id")) and rec.get("candidate_id") == seed.get("candidate_id"))
    feasible_ids = {x.get("candidate_id") for x in evidence_object.get("E_ranked_candidates_top", []) if isinstance(x, dict)}
    feasible_recommendation = float(rec.get("candidate_id") in feasible_ids) if rec.get("candidate_id") else 0.0

    covered_risks = used_risks & allowed_risks
    risk_recall = len(covered_risks) / len(allowed_risks) if allowed_risks else 1.0
    evidence_numbers = all_numeric_evidence_values(evidence_object)
    claim_numbers = numerical_claim_values(report, evidence_object) if isinstance(report, dict) else []
    suspicious = sorted(set(value for value in claim_numbers if not number_supported(value, evidence_numbers)))
    numerical_conflict_rate = len(suspicious) / len(set(claim_numbers)) if claim_numbers else 0.0

    verifier_pass = (
        not schema_errs
        and not invalid_evidence
        and not invalid_risks
        and evidence_coverage == 1.0
        and risk_recall == 1.0
        and recommendation_consistency == 1.0
        and feasible_recommendation == 1.0
        and not suspicious
    )
    return {
        "parse_success": float(isinstance(report, dict)),
        "parse_status": parse_status,
        "schema_valid": float(not schema_errs),
        "schema_completeness": schema_completeness,
        "schema_errors": schema_errs,
        "evidence_coverage": evidence_coverage,
        "unsupported_claim_rate": 1.0 - evidence_coverage,
        "used_evidence_id_count": len(used_evidence),
        "invalid_evidence_id_count": len(invalid_evidence),
        "invalid_evidence_ids": invalid_evidence,
        "used_risk_item_id_count": len(used_risks),
        "invalid_risk_item_id_count": len(invalid_risks),
        "invalid_risk_item_ids": invalid_risks,
        "required_risk_count": len(allowed_risks),
        "covered_required_risk_count": len(covered_risks),
        "missing_required_risk_count": len(allowed_risks - covered_risks),
        "missing_required_risk_ids": sorted(allowed_risks - covered_risks),
        "risk_recall": risk_recall,
        "numerical_conflict_count": len(suspicious),
        "numerical_conflict_rate": numerical_conflict_rate,
        "suspicious_numbers": suspicious,
        "recommendation_consistency": recommendation_consistency,
        "feasible_recommendation": feasible_recommendation,
        "verifier_pass": float(verifier_pass),
    }


In [ ]:
def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value)).strip("_")[:80] or "unknown"

def condition_of(package):
    return package.get("joint_requirements", {}).get("condition", "unknown")

def experiment_config_hash():
    config = {
        "model": DOGAPI_MODEL, "temperature": TEMPERATURE, "top_p": TOP_P,
        "max_tokens": MAX_TOKENS, "n_repeats": N_REPEATS,
        "groups": GROUPS, "schema": REPORT_SCHEMA,
    }
    return sha256_text(canonical_json(config))

CONFIG_HASH = experiment_config_hash()

def result_path(package_index, condition, group, repeat):
    return OUTPUT_DIR / f"P{package_index:02d}_{safe_name(DOGAPI_MODEL)}_{group}_{safe_name(condition)}_{repeat:02d}.json"

def successful_payload(payload):
    return (
        payload.get("config_hash") == CONFIG_HASH
        and payload.get("model") == DOGAPI_MODEL
        and bool(payload.get("raw_text"))
        and not bool(payload.get("api_error"))
    )

def load_completed():
    completed, ignored = {}, []
    for path in sorted(OUTPUT_DIR.glob("P*.json")):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            ignored.append((path.name, "json_parse_error"))
            continue
        if not successful_payload(payload):
            ignored.append((path.name, "incompatible_or_failed"))
            continue
        key = (payload.get("condition"), payload.get("group"), int(payload.get("repeat")))
        completed[key] = path
    return completed, ignored

def call_llm(system_prompt, user_prompt):
    from openai import OpenAI
    if not os.getenv("DOGAPI_API_KEY"):
        os.environ["DOGAPI_API_KEY"] = getpass.getpass("Enter the DogAPI API key: ")
    client = OpenAI(api_key=os.environ["DOGAPI_API_KEY"], base_url=DOGAPI_BASE_URL, timeout=REQUEST_TIMEOUT)
    return client.chat.completions.create(
        model=DOGAPI_MODEL,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
        temperature=TEMPERATURE,
        top_p=TOP_P,
        max_tokens=MAX_TOKENS,
    ).choices[0].message.content

def generate_once(package_index, package, evidence_object, group, repeat, order_index):
    system_prompt, user_prompt, evidence_json, schema_json = build_prompts(evidence_object, group)
    raw_text, api_error, transport_attempts = None, None, []
    for attempt in range(1, NETWORK_RETRIES + 1):
        try:
            raw_text = call_llm(system_prompt, user_prompt)
            transport_attempts.append({"attempt": attempt, "success": True})
            break
        except Exception as exc:
            api_error = {"error_type": type(exc).__name__, "error_message": str(exc)[:1000]}
            transport_attempts.append({"attempt": attempt, "success": False, **api_error})
            if attempt < NETWORK_RETRIES:
                time.sleep(NETWORK_BACKOFF_SECONDS * attempt)

    report, parse_status = parse_report(raw_text)
    verifier = verify_report(report, raw_text, evidence_object, parse_status)
    condition = condition_of(package)
    payload = {
        "experiment": "21_fair_llm_comparison",
        "run_id": str(uuid.uuid4()),
        "timestamp": datetime.now().isoformat(),
        "package_index": package_index,
        "condition": condition,
        "group": group,
        "repeat": repeat,
        "randomized_call_order": order_index,
        "model": DOGAPI_MODEL,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": MAX_TOKENS,
        "config_hash": CONFIG_HASH,
        "evidence_object_sha256": sha256_text(evidence_json),
        "schema_sha256": sha256_text(schema_json),
        "common_task_sha256": sha256_text(COMMON_TASK),
        "group_rule_sha256": sha256_text(GROUP_RULES[group]),
        "transport_attempts": transport_attempts,
        "api_error": api_error if raw_text is None else None,
        "raw_text": raw_text,
        "report": report,
        "verifier": verifier,
    }
    result_path(package_index, condition, group, repeat).write_text(
        json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    return payload

completed, ignored = load_completed()
print(f"Compatible successful results detected: {len(completed)}/{expected_total}; only missing runs will be generated.")
if ignored:
    print("Ignored files from prior models/configurations or failed runs:", pd.Series([reason for _, reason in ignored]).value_counts().to_dict())

if RUN_LLM_CALLS:
    tasks = []
    rng = random.Random(RANDOM_STATE)
    for package_index, (package, evidence_object) in enumerate(zip(packages, evidence_objects), 1):
        condition = condition_of(package)
        for repeat in range(1, N_REPEATS + 1):
            order = GROUPS.copy()
            rng.shuffle(order)  # Interleave the randomized call order to reduce temporal-order bias.
            for order_index, group in enumerate(order, 1):
                tasks.append((package_index, package, evidence_object, group, repeat, order_index))

    skipped = called = failed = 0
    for package_index, package, evidence_object, group, repeat, order_index in tasks:
        condition = condition_of(package)
        key = (condition, group, repeat)
        if key in completed:
            skipped += 1
            print("SKIP", condition, group, repeat)
            continue
        print("CALL", condition, group, repeat, f"order={order_index}")
        payload = generate_once(package_index, package, evidence_object, group, repeat, order_index)
        called += 1
        if successful_payload(payload):
            completed[key] = result_path(package_index, condition, group, repeat)
        else:
            failed += 1
            print("FAILED", payload.get("api_error"))
    print(f"Completed: skipped={skipped}, new calls={called}, failed={failed}, successful={len(completed)}/{expected_total}.")
else:
    print("No API calls were made. For a new generation run, set RUN_LLM_CALLS to True in the second notebook cell and rerun from this cell.")


In [ ]:
def bootstrap_mean_ci(values, n_boot=5000, seed=RANDOM_STATE):
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if not len(values):
        return np.nan, np.nan
    rng = np.random.default_rng(seed)
    means = np.mean(rng.choice(values, size=(n_boot, len(values)), replace=True), axis=1)
    return tuple(np.quantile(means, [0.025, 0.975]))

def paired_sign_flip_pvalue(differences, n_perm=20000, seed=RANDOM_STATE):
    differences = np.asarray(differences, float)
    differences = differences[np.isfinite(differences)]
    if not len(differences):
        return np.nan
    observed = abs(differences.mean())
    rng = np.random.default_rng(seed)
    simulated = np.empty(n_perm)
    for i in range(n_perm):
        simulated[i] = abs(np.mean(differences * rng.choice([-1.0, 1.0], size=len(differences))))
    return float((1 + np.sum(simulated >= observed)) / (n_perm + 1))

payloads = []
for path in sorted(OUTPUT_DIR.glob("P*.json")):
    try:
        payload = json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        continue
    if payload.get("config_hash") == CONFIG_HASH and payload.get("group") in GROUPS:
        payload["_source_file"] = path.name
        payloads.append(payload)

rows = []
for payload in payloads:
    package_index = int(payload.get("package_index"))
    evidence_object = evidence_objects[package_index - 1]
    report = payload.get("report")
    raw_text = payload.get("raw_text")
    parse_status = (payload.get("verifier") or {}).get("parse_status") or ("parsed" if isinstance(report, dict) else "parse_error")
    verifier = verify_report(report, raw_text, evidence_object, parse_status)
    rows.append({
        "condition": payload.get("condition"), "group": payload.get("group"), "repeat": payload.get("repeat"),
        "run_id": payload.get("run_id"), "source_file": payload.get("_source_file"),
        "api_error": float(bool(payload.get("api_error"))), "raw_text_nonempty": float(bool(raw_text)),
        **{key: value for key, value in verifier.items() if not isinstance(value, (list, dict))},
    })

metrics = pd.DataFrame(rows)
metric_names = [
    "parse_success", "schema_valid", "schema_completeness", "evidence_coverage", "unsupported_claim_rate",
    "invalid_evidence_id_count", "invalid_risk_item_id_count", "risk_recall", "numerical_conflict_count",
    "numerical_conflict_rate", "recommendation_consistency", "feasible_recommendation", "verifier_pass",
]

if metrics.empty:
    print(f"No results match the active configuration (0/{expected_total}). Enable RUN_LLM_CALLS to generate them, or point OUTPUT_DIR_OVERRIDE to compatible prior results.")
    summary = pd.DataFrame()
    paired = pd.DataFrame()
else:
    metrics.to_csv(OUTPUT_DIR / "automatic_metrics_per_generation.csv", index=False, encoding="utf-8-sig")
    valid = metrics[(metrics["api_error"] == 0) & (metrics["raw_text_nonempty"] == 1)].copy()
    summary_rows = []
    for group, part in valid.groupby("group"):
        row = {"group": group, "n_valid": len(part), "n_conditions": part["condition"].nunique()}
        for metric in metric_names:
            values = pd.to_numeric(part[metric], errors="coerce")
            lo, hi = bootstrap_mean_ci(values)
            row[f"{metric}_mean"] = values.mean()
            row[f"{metric}_std"] = values.std(ddof=1)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        summary_rows.append(row)
    summary = pd.DataFrame(summary_rows)

    paired_rows = []
    index_cols = ["condition", "repeat"]
    for metric in metric_names:
        table = valid.pivot_table(index=index_cols, columns="group", values=metric, aggfunc="first").dropna()
        if set(GROUPS).issubset(table.columns):
            difference = table["ec_llm"].to_numpy(float) - table["ordinary_llm"].to_numpy(float)
            lo, hi = bootstrap_mean_ci(difference)
            sd = np.std(difference, ddof=1)
            paired_rows.append({
                "metric": metric, "n_pairs": len(difference),
                "ordinary_mean": table["ordinary_llm"].mean(), "ec_llm_mean": table["ec_llm"].mean(),
                "paired_difference_ec_minus_ordinary": difference.mean(),
                "paired_difference_ci95_low": lo, "paired_difference_ci95_high": hi,
                "paired_standardized_effect": difference.mean() / sd if sd > 0 else np.nan,
                "paired_sign_flip_pvalue": paired_sign_flip_pvalue(difference),
            })
    paired = pd.DataFrame(paired_rows)

    summary.to_csv(OUTPUT_DIR / "automatic_metrics_summary_with_ci.csv", index=False, encoding="utf-8-sig")
    paired.to_csv(OUTPUT_DIR / "paired_comparison_statistics.csv", index=False, encoding="utf-8-sig")
    coverage = valid.groupby(["condition", "group"]).size().rename("n").reset_index()
    coverage.to_csv(OUTPUT_DIR / "condition_group_coverage.csv", index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(OUTPUT_DIR / "21_fair_llm_comparison_results.xlsx", engine="openpyxl") as writer:
        metrics.to_excel(writer, sheet_name="per_generation", index=False)
        summary.to_excel(writer, sheet_name="summary_ci", index=False)
        paired.to_excel(writer, sheet_name="paired_statistics", index=False)
        coverage.to_excel(writer, sheet_name="coverage", index=False)

    display(summary)
    display(paired)

    plot_metrics = ["schema_valid", "evidence_coverage", "risk_recall", "recommendation_consistency", "verifier_pass"]
    plot_data = valid.groupby("group")[plot_metrics].mean().reindex(GROUPS)
    plot_data = plot_data.rename(index={"ordinary_llm": "Ordinary LLM", "ec_llm": "EC-LLM"})
    ax = plot_data.T.plot(kind="bar", figsize=(10, 4.8), color=["#777777", "#247a63"])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Mean")
    ax.set_xlabel("")
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "automatic_metrics_group_comparison.png", dpi=240, bbox_inches="tight")
    plt.show()

print("Automatic-evaluation output directory:", OUTPUT_DIR)


In [ ]:
# Generate a stratified, randomized, anonymized expert-review template with recoverable group labels.
successful = [p for p in payloads if p.get("raw_text") and not p.get("api_error")]
selection_rng = np.random.default_rng(RANDOM_STATE)
selected_payloads = []

if successful:
    payload_frame = pd.DataFrame([{
        "index": i, "condition": p.get("condition"), "group": p.get("group"), "repeat": p.get("repeat")
    } for i, p in enumerate(successful)])
    for (_, _), part in payload_frame.groupby(["condition", "group"]):
        n = min(HUMAN_SAMPLES_PER_CONDITION_GROUP, len(part))
        chosen = selection_rng.choice(part["index"].to_numpy(), size=n, replace=False)
        selected_payloads.extend(successful[int(i)] for i in chosen)

blind_rows, key_rows = [], []
for payload in selected_payloads:
    blind_id = hashlib.sha256(f"{payload['run_id']}|blind|{RANDOM_STATE}".encode()).hexdigest()[:10]
    key_rows.append({
        "blind_id": blind_id, "run_id": payload.get("run_id"), "condition": payload.get("condition"),
        "group": payload.get("group"), "repeat": payload.get("repeat"), "source_file": payload.get("_source_file"),
    })
    for rater_id in RATER_IDS:
        blind_rows.append({
            "blind_id": blind_id, "condition": payload.get("condition"), "rater_id": rater_id,
            "text": payload.get("raw_text"), "clarity_1to5": "", "evidence_fidelity_1to5": "",
            "risk_actionability_1to5": "", "auditability_1to5": "", "overall_usefulness_1to5": "", "comments": "",
        })

blind_template = pd.DataFrame(blind_rows)
if not blind_template.empty:
    blind_template = blind_template.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
blind_key = pd.DataFrame(key_rows).drop_duplicates("blind_id") if key_rows else pd.DataFrame()
blind_template.to_excel(OUTPUT_DIR / "human_blind_review_template.xlsx", index=False)
blind_key.to_csv(OUTPUT_DIR / "human_blind_review_key_DO_NOT_SHARE_WITH_RATERS.csv", index=False, encoding="utf-8-sig")

print("Blind-review template rows:", len(blind_template), "; anonymized texts:", len(blind_key))
print("Provide reviewers only with human_blind_review_template.xlsx; do not provide the group-label key file.")


In [ ]:
# Rerun this cell after expert review to restore group labels and summarize the ratings.
review_path = OUTPUT_DIR / "human_blind_review_template.xlsx"
review = pd.read_excel(review_path) if review_path.exists() else pd.DataFrame()
score_columns = ["clarity_1to5", "evidence_fidelity_1to5", "risk_actionability_1to5", "auditability_1to5", "overall_usefulness_1to5"]

if review.empty or not all(pd.to_numeric(review[col], errors="coerce").notna().all() for col in score_columns):
    print("Expert ratings are incomplete; subjective-evaluation results will not be generated.")
else:
    key = pd.read_csv(OUTPUT_DIR / "human_blind_review_key_DO_NOT_SHARE_WITH_RATERS.csv")
    scored = review.merge(key[["blind_id", "group", "run_id", "repeat"]], on="blind_id", how="left", validate="many_to_one")
    for col in score_columns:
        scored[col] = pd.to_numeric(scored[col], errors="raise")
        if not scored[col].between(1, 5).all():
            raise ValueError(f"{col} contains ratings outside the permitted 1-5 range.")

    human_summary = scored.groupby("group")[score_columns].agg(["mean", "std", "count"])
    human_summary.to_csv(OUTPUT_DIR / "human_review_summary.csv", encoding="utf-8-sig")
    scored.to_csv(OUTPUT_DIR / "human_review_scored_with_hidden_group.csv", index=False, encoding="utf-8-sig")

    # With at least two reviewers, report pairwise quadratic-weighted Cohen kappa.
    from sklearn.metrics import cohen_kappa_score
    kappa_rows = []
    raters = sorted(scored["rater_id"].unique())
    for metric in score_columns:
        wide = scored.pivot_table(index="blind_id", columns="rater_id", values=metric, aggfunc="first")
        for i, r1 in enumerate(raters):
            for r2 in raters[i + 1:]:
                pair = wide[[r1, r2]].dropna()
                kappa_rows.append({
                    "metric": metric, "rater_1": r1, "rater_2": r2, "n": len(pair),
                    "quadratic_weighted_cohen_kappa": cohen_kappa_score(pair[r1], pair[r2], weights="quadratic") if len(pair) else np.nan,
                })
    kappas = pd.DataFrame(kappa_rows)
    kappas.to_csv(OUTPUT_DIR / "human_review_interrater_kappa.csv", index=False, encoding="utf-8-sig")
    display(human_summary)
    display(kappas)


## Manuscript reporting requirements

1. The primary comparison reports only the ordinary LLM and EC-LLM; RAG or regeneration must not be introduced as a third experimental group.
2. Apply the same offline verifier to both groups and report the valid sample count for each group. Exclude API failures from quality-metric denominators, but report the failure rate separately.
3. Define each pair by the same task condition and repeat index. Report the mean EC-LLM-minus-ordinary-LLM difference, its 95% bootstrap confidence interval, and the paired sign-flip test.
4. The numerical-conflict rate is a heuristic automated metric. Interpret it jointly with expert evidence-fidelity ratings rather than as sufficient proof of factual correctness.
5. Do not provide the group-label key to reviewers after generating the blind-review template. Report reviewer count, expertise, rating scale, and inter-rater agreement in the manuscript.
6. EC-LLM is evaluated using single-pass outputs in this experiment. Verifier-driven regeneration may be described as a deployment safeguard but must not be mixed into the primary comparison.
